In [1]:
import os
import random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import time
import copy
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score
)

In [2]:
SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 15
LR = 1e-4
WEIGHT_DECAY = 1e-3
DEVICE = torch.device('cpu')

TRAIN_DIR = r'D:\ScanO-assign-Apaar\dataset\train'
TEST_DIR = r'D:\ScanO-assign-Apaar\dataset\test'
OUTPUT_DIR = 'submission/outputs'

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/sample_outputs', exist_ok=True)

In [3]:
def build_df(root):
    paths = []
    labels = []

    for label, cls in enumerate(['normal', 'pneumonia']):
        cls_path = os.path.join(root, cls)

        for img in os.listdir(cls_path):
            paths.append(os.path.join(cls_path, img))
            labels.append(label)

    return pd.DataFrame({
        'image_path': paths,
        'label': labels
    })

train_df = build_df(TRAIN_DIR)

In [4]:
train_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df['label'],
    random_state=SEED
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

In [5]:
train_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [6]:
class XrayDataset(Dataset):
    def __init__(self, df, tfms):
        self.df = df
        self.tfms = tfms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img = Image.open(row.image_path).convert('RGB')
        img = self.tfms(img)

        label = torch.tensor(row.label).long()

        return img, label

In [7]:
train_ds = XrayDataset(train_df, train_tfms)
val_ds = XrayDataset(val_df, val_tfms)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

In [8]:
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

model.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(512, 2)
)

model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

In [9]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.3,
    patience=2
)

In [10]:
best_acc = 0
best_loss = np.inf

for epoch in range(EPOCHS):

    model.train()
    train_loss = 0

    for imgs, labels in tqdm(train_loader):

        imgs = imgs.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(imgs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    model.eval()

    val_loss = 0

    preds = []
    trues = []

    with torch.no_grad():

        for imgs, labels in val_loader:

            imgs = imgs.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(imgs)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            p = outputs.argmax(1)

            preds.extend(p.cpu().numpy())
            trues.extend(labels.cpu().numpy())

    acc = accuracy_score(trues, preds)

    print(f'Epoch {epoch+1}')
    print(f'Train Loss: {train_loss/len(train_loader):.4f}')
    print(f'Val Loss: {val_loss/len(val_loader):.4f}')
    print(f'Val Acc : {acc:.4f}')

    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), f'{OUTPUT_DIR}/best_model_phase3.pth')

100%|██████████| 32/32 [01:07<00:00,  2.11s/it]


Epoch 1
Train Loss: 0.4513
Val Loss: 0.4645
Val Acc : 0.8672


100%|██████████| 32/32 [01:02<00:00,  1.96s/it]


Epoch 2
Train Loss: 0.3847
Val Loss: 0.4490
Val Acc : 0.8828


100%|██████████| 32/32 [01:00<00:00,  1.90s/it]


...

Epoch 13
Train Loss: 0.2847
Val Loss: 0.4749
Val Acc : 0.9062


100%|██████████| 32/32 [01:53<00:00,  3.55s/it]


Epoch 14
Train Loss: 0.2615
Val Loss: 0.4325
Val Acc : 0.8750


100%|██████████| 32/32 [01:42<00:00,  3.21s/it]


Epoch 15
Train Loss: 0.2657
Val Loss: 0.4618
Val Acc : 0.8828


In [11]:
# ── Training Loop ─────────────────────────────────────────────
best_loss  = float('inf')
best_acc   = 0.0
best_state = None
history    = []

start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for imgs, labels in tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}'):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        preds         = outputs.argmax(dim=1)
        correct      += (preds == labels).sum().item()
        total        += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc  = correct / total
    scheduler.step(epoch_loss)
    history.append((epoch, epoch_loss, epoch_acc))

    print(f'Epoch {epoch:02d} | Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f}')

    if epoch_loss < best_loss:
        best_loss  = epoch_loss
        best_acc   = epoch_acc
        best_state = copy.deepcopy(model.state_dict())
        print(f'  --> Best model saved (loss={best_loss:.4f})')

elapsed = time.time() - start_time
print(f'\nTraining complete in {elapsed/60:.1f} min')

Epoch 1/15: 100%|██████████| 32/32 [01:40<00:00,  3.13s/it]


Epoch 01 | Loss: 0.2545 | Acc: 0.9824
  --> Best model saved (loss=0.2545)


Epoch 2/15: 100%|██████████| 32/32 [01:49<00:00,  3.41s/it]


Epoch 02 | Loss: 0.2452 | Acc: 0.9922
  --> Best model saved (loss=0.2452)


Epoch 3/15: 100%|██████████| 32/32 [01:40<00:00,  3.13s/it]


...

Epoch 13 | Loss: 0.2308 | Acc: 0.9941


Epoch 14/15: 100%|██████████| 32/32 [01:34<00:00,  2.95s/it]


Epoch 14 | Loss: 0.2223 | Acc: 1.0000
  --> Best model saved (loss=0.2223)


Epoch 15/15: 100%|██████████| 32/32 [01:36<00:00,  3.02s/it]

Epoch 15 | Loss: 0.2198 | Acc: 1.0000
  --> Best model saved (loss=0.2198)

Training complete in 25.0 min


In [12]:
# ── Save best model ───────────────────────────────────────────
save_path = os.path.join(OUTPUT_DIR, 'best_model_phase3.pth')
torch.save(best_state, save_path)
print(f'Saved: {save_path}')
print(f'Best train loss : {best_loss:.4f}')
print(f'Best train acc  : {best_acc:.4f}')

Saved: submission/outputs\best_model_phase3.pth
Best train loss : 0.2198
Best train acc  : 1.0000
